In [1]:
#%conda install PyVCF

In [2]:
import io
import os
import pandas as pd
import re
import vcf

In [3]:
def read_vcf(path):
    with open(path, 'r') as f:
        lines = [l for l in f if not l.startswith('##')]
    return pd.read_csv(
        io.StringIO(''.join(lines)),
        dtype={'#CHROM': str, 'POS': int, 'ID': str, 'REF': str, 'ALT': str,
               'QUAL': str, 'FILTER': str, 'INFO': str},
        sep='\t'
    ).rename(columns={'#CHROM': 'CHROM'})

def extract_af(info):
    match = re.search(r'AF=([\d\.]+)', info)
    return match.group(1) if match else None

def filter_vcf(vcf_file):
    vcf_reader = vcf.Reader(open(vcf_file, 'r'))

    for record in vcf_reader:
        if 'AF' in record.INFO and record.INFO['AF'] > 0.5:
            print(record)

def fasta_to_all_chars_dataframe(filepath):
    """ 
    Parses a FASTA file and returns a DataFrame with all characters enumerated
    """
    with open(filepath, 'r') as file:
        position = 1  
        data = []  
        for line in file:
            for char in line:
                if char == '\n':
                    data.append((position, 'line_break', '\\n'))
                else:
                    data.append((position, 'sequence' if char != '>' else 'header', char))
                position += 1

    return pd.DataFrame(data, columns=['POS', 'Type', 'REF'])

def find_deletions(row):
    """ 
    Determines deletions
    """
    pos = row['POS']
    ref = row['REF']
    alt = row['ALT']
    deletions = []
    start_deletion = pos + len(alt)
    # Subtract the ALT length from REF to determine the deletions
    extra_bases = ref[len(alt):]
    for i, base in enumerate(extra_bases, start=start_deletion):
        deletions.append((i, base))
        
    return deletions

def replace_ref(row):
    if  row['REF_x'] == row['Base']:
        return 0
    else:
        return row['REF_x']

In [4]:
df = read_vcf('/Users/fionachow/Documents/NYU/CDS/Spring 2024/Research Fair/Hochwagen/Individual Data/ERR3240115/ERR3240115_rDNA.vcf') #vcf file from original variant calling of 1 individual

df.head(30)

,CHROM,POS,ID,REF,ALT,QUAL,FILTER,INFO
0,Human,31,.,T,C,21247,PASS,"DP=1510;AF=1.000000;SB=0;DP4=0,0,1509,1"
1,Human,32,.,C,T,22076,PASS,"DP=1554;AF=1.000000;SB=0;DP4=0,0,1553,1"
2,Human,73,.,C,A,46496,PASS,"DP=2625;AF=0.996952;SB=0;DP4=7,0,2607,10"
3,Human,74,.,A,C,48045,PASS,"DP=2657;AF=0.999624;SB=0;DP4=1,0,2646,10"
4,Human,80,.,A,AC,49314,PASS,"DP=2829;AF=0.977024;SB=0;DP4=64,0,2747,17;INDE..."
5,Human,103,.,GC,G,49314,PASS,"DP=3376;AF=0.987559;SB=31;DP4=38,4,3304,30;IND..."
6,Human,121,.,C,G,49314,PASS,"DP=3853;AF=0.998962;SB=0;DP4=0,0,3779,70"
7,Human,209,.,C,G,2469,PASS,"DP=4542;AF=0.109203;SB=2;DP4=3584,459,436,60"
8,Human,237,.,G,GC,49314,PASS,"DP=4988;AF=0.966921;SB=14;DP4=164,21,3988,835;..."
9,Human,270,.,C,CG,49314,PASS,"DP=5712;AF=0.945553;SB=6;DP4=242,88,4122,1279;..."


In [5]:
df[df['REF'].str.len() > 1] #checks for reference alleles having more than 1 base. Why does lofreq report 'REF' this way?

,CHROM,POS,ID,REF,ALT,QUAL,FILTER,INFO
5,Human,103,.,GC,G,49314,PASS,"DP=3376;AF=0.987559;SB=31;DP4=38,4,3304,30;IND..."
19,Human,537,.,TG,T,49314,PASS,"DP=9753;AF=0.957346;SB=86;DP4=310,144,5049,428..."
24,Human,633,.,GT,G,529,PASS,"DP=9666;AF=0.052245;SB=10;DP4=4846,4397,284,22..."
28,Human,763,.,CGG,C,49314,PASS,"DP=9684;AF=0.649318;SB=75;DP4=1732,1687,2816,3..."
31,Human,963,.,TGAGAC,T,394,PASS,"DP=9244;AF=0.008979;SB=14;DP4=4474,4750,50,33;..."
...,...,...,...,...,...,...,...,...
318,Human,11069,.,GC,G,28436,PASS,"DP=2541;AF=0.935852;SB=2;DP4=66,133,838,1540;I..."
323,Human,11118,.,GC,G,17778,PASS,"DP=2197;AF=0.863450;SB=0;DP4=161,173,931,966;I..."
329,Human,11268,.,CG,C,19335,PASS,"DP=3057;AF=0.718024;SB=1045;DP4=790,104,1061,1..."
333,Human,11417,.,GGCA,G,3864,PASS,"DP=3769;AF=0.205890;SB=0;DP4=1936,1082,496,280..."


In [6]:
df[(df['REF'].str.len()) > (df['ALT'].str.len())] #deletion instances

,CHROM,POS,ID,REF,ALT,QUAL,FILTER,INFO
5,Human,103,.,GC,G,49314,PASS,"DP=3376;AF=0.987559;SB=31;DP4=38,4,3304,30;IND..."
19,Human,537,.,TG,T,49314,PASS,"DP=9753;AF=0.957346;SB=86;DP4=310,144,5049,428..."
24,Human,633,.,GT,G,529,PASS,"DP=9666;AF=0.052245;SB=10;DP4=4846,4397,284,22..."
28,Human,763,.,CGG,C,49314,PASS,"DP=9684;AF=0.649318;SB=75;DP4=1732,1687,2816,3..."
31,Human,963,.,TGAGAC,T,394,PASS,"DP=9244;AF=0.008979;SB=14;DP4=4474,4750,50,33;..."
...,...,...,...,...,...,...,...,...
318,Human,11069,.,GC,G,28436,PASS,"DP=2541;AF=0.935852;SB=2;DP4=66,133,838,1540;I..."
323,Human,11118,.,GC,G,17778,PASS,"DP=2197;AF=0.863450;SB=0;DP4=161,173,931,966;I..."
329,Human,11268,.,CG,C,19335,PASS,"DP=3057;AF=0.718024;SB=1045;DP4=790,104,1061,1..."
333,Human,11417,.,GGCA,G,3864,PASS,"DP=3769;AF=0.205890;SB=0;DP4=1936,1082,496,280..."


In [7]:
df[(df['REF'].str.len()) < (df['ALT'].str.len())] #insertion instances

,CHROM,POS,ID,REF,ALT,QUAL,FILTER,INFO
4,Human,80,.,A,AC,49314,PASS,"DP=2829;AF=0.977024;SB=0;DP4=64,0,2747,17;INDE..."
8,Human,237,.,G,GC,49314,PASS,"DP=4988;AF=0.966921;SB=14;DP4=164,21,3988,835;..."
9,Human,270,.,C,CG,49314,PASS,"DP=5712;AF=0.945553;SB=6;DP4=242,88,4122,1279;..."
10,Human,275,.,C,CG,49314,PASS,"DP=5882;AF=0.937946;SB=16;DP4=327,82,4131,1386..."
11,Human,284,.,A,AC,49314,PASS,"DP=6096;AF=0.960138;SB=15;DP4=185,91,4284,1569..."
...,...,...,...,...,...,...,...,...
372,Human,13113,.,C,CG,46985,PASS,"DP=2411;AF=0.950643;SB=164;DP4=66,78,344,1948;..."
373,Human,13150,.,A,AC,49314,PASS,"DP=2781;AF=0.961884;SB=10;DP4=11,103,152,2523;..."
374,Human,13219,.,C,CG,43670,PASS,"DP=2182;AF=0.944088;SB=1;DP4=1,152,29,2031;IND..."
376,Human,13278,.,G,GGC,13133,PASS,"DP=1033;AF=0.909971;SB=0;DP4=0,103,2,938;INDEL..."


In [8]:
#extracting AF into a seperate column
df['AF'] = df['INFO'].apply(extract_af)

df

,CHROM,POS,ID,REF,ALT,QUAL,FILTER,INFO,AF
0,Human,31,.,T,C,21247,PASS,"DP=1510;AF=1.000000;SB=0;DP4=0,0,1509,1",1.000000
1,Human,32,.,C,T,22076,PASS,"DP=1554;AF=1.000000;SB=0;DP4=0,0,1553,1",1.000000
2,Human,73,.,C,A,46496,PASS,"DP=2625;AF=0.996952;SB=0;DP4=7,0,2607,10",0.996952
3,Human,74,.,A,C,48045,PASS,"DP=2657;AF=0.999624;SB=0;DP4=1,0,2646,10",0.999624
4,Human,80,.,A,AC,49314,PASS,"DP=2829;AF=0.977024;SB=0;DP4=64,0,2747,17;INDE...",0.977024
...,...,...,...,...,...,...,...,...,...
374,Human,13219,.,C,CG,43670,PASS,"DP=2182;AF=0.944088;SB=1;DP4=1,152,29,2031;IND...",0.944088
375,Human,13235,.,GC,G,35659,PASS,"DP=1803;AF=0.953411;SB=22;DP4=5,91,18,1701;IND...",0.953411
376,Human,13278,.,G,GGC,13133,PASS,"DP=1033;AF=0.909971;SB=0;DP4=0,103,2,938;INDEL...",0.909971
377,Human,13280,.,G,C,13572,PASS,"DP=1017;AF=1.000000;SB=0;DP4=0,0,2,1015",1.000000


In [9]:
type(df.loc[2, 'AF']) #string type

str

In [10]:
df['AF'] = df['AF'].astype(float) #converting format to float

type(df.loc[2, 'AF'])

numpy.float64

In [11]:
filtered_df = df[df['AF'] > 0.5] #filtering for AF threshold >0.5; extracting majority alleles at each position from variant calling

filtered_df

,CHROM,POS,ID,REF,ALT,QUAL,FILTER,INFO,AF
0,Human,31,.,T,C,21247,PASS,"DP=1510;AF=1.000000;SB=0;DP4=0,0,1509,1",1.000000
1,Human,32,.,C,T,22076,PASS,"DP=1554;AF=1.000000;SB=0;DP4=0,0,1553,1",1.000000
2,Human,73,.,C,A,46496,PASS,"DP=2625;AF=0.996952;SB=0;DP4=7,0,2607,10",0.996952
3,Human,74,.,A,C,48045,PASS,"DP=2657;AF=0.999624;SB=0;DP4=1,0,2646,10",0.999624
4,Human,80,.,A,AC,49314,PASS,"DP=2829;AF=0.977024;SB=0;DP4=64,0,2747,17;INDE...",0.977024
...,...,...,...,...,...,...,...,...,...
374,Human,13219,.,C,CG,43670,PASS,"DP=2182;AF=0.944088;SB=1;DP4=1,152,29,2031;IND...",0.944088
375,Human,13235,.,GC,G,35659,PASS,"DP=1803;AF=0.953411;SB=22;DP4=5,91,18,1701;IND...",0.953411
376,Human,13278,.,G,GGC,13133,PASS,"DP=1033;AF=0.909971;SB=0;DP4=0,103,2,938;INDEL...",0.909971
377,Human,13280,.,G,C,13572,PASS,"DP=1017;AF=1.000000;SB=0;DP4=0,0,2,1015",1.000000


In [12]:
temp_deletion = filtered_df[(filtered_df['REF'].str.len()) > (filtered_df['ALT'].str.len())] # deletion instances that matter

display(temp_deletion)

,CHROM,POS,ID,REF,ALT,QUAL,FILTER,INFO,AF
5,Human,103,.,GC,G,49314,PASS,"DP=3376;AF=0.987559;SB=31;DP4=38,4,3304,30;IND...",0.987559
19,Human,537,.,TG,T,49314,PASS,"DP=9753;AF=0.957346;SB=86;DP4=310,144,5049,428...",0.957346
28,Human,763,.,CGG,C,49314,PASS,"DP=9684;AF=0.649318;SB=75;DP4=1732,1687,2816,3...",0.649318
53,Human,1671,.,GCTCCC,G,12978,PASS,"DP=7755;AF=0.596132;SB=1;DP4=1326,1605,2116,25...",0.596132
58,Human,1756,.,GT,G,49314,PASS,"DP=8493;AF=0.947604;SB=0;DP4=217,287,3509,4539...",0.947604
81,Human,2486,.,TGCG,T,49314,PASS,"DP=8024;AF=0.542124;SB=3;DP4=1762,1936,2033,23...",0.542124
103,Human,3152,.,GC,G,49314,PASS,"DP=3938;AF=0.949975;SB=23;DP4=73,155,1555,2186...",0.949975
111,Human,3411,.,GC,G,49314,PASS,"DP=5172;AF=0.728925;SB=0;DP4=998,426,2634,1136...",0.728925
157,Human,5293,.,AG,A,49314,PASS,"DP=8637;AF=0.957393;SB=32;DP4=245,197,3874,439...",0.957393
158,Human,5303,.,CG,C,49314,PASS,"DP=8804;AF=0.950704;SB=36;DP4=187,295,3975,439...",0.950704


In [13]:
# Applying the function to each row
temp_deletion['Deletions'] = temp_deletion.apply(find_deletions, axis=1)

# Exploding the list to have each deletion in a separate row
deletions_expanded = temp_deletion.explode('Deletions')
deletions_expanded[['Delete_Position', 'Base']] = pd.DataFrame(deletions_expanded['Deletions'].tolist(), index=deletions_expanded.index)

display(deletions_expanded)

/var/folders/hb/l21bg8lx3tgc5ky3s56cwdth0000gn/T/ipykernel_29326/2052078697.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  temp_deletion['Deletions'] = temp_deletion.apply(find_deletions, axis=1)


,CHROM,POS,ID,REF,ALT,QUAL,FILTER,INFO,AF,Deletions,Delete_Position,Base
5,Human,103,.,GC,G,49314,PASS,"DP=3376;AF=0.987559;SB=31;DP4=38,4,3304,30;IND...",0.987559,"(104, C)",104,C
19,Human,537,.,TG,T,49314,PASS,"DP=9753;AF=0.957346;SB=86;DP4=310,144,5049,428...",0.957346,"(538, G)",538,G
28,Human,763,.,CGG,C,49314,PASS,"DP=9684;AF=0.649318;SB=75;DP4=1732,1687,2816,3...",0.649318,"(764, G)",764,G
28,Human,763,.,CGG,C,49314,PASS,"DP=9684;AF=0.649318;SB=75;DP4=1732,1687,2816,3...",0.649318,"(765, G)",765,G
53,Human,1671,.,GCTCCC,G,12978,PASS,"DP=7755;AF=0.596132;SB=1;DP4=1326,1605,2116,25...",0.596132,"(1672, C)",1672,C
53,Human,1671,.,GCTCCC,G,12978,PASS,"DP=7755;AF=0.596132;SB=1;DP4=1326,1605,2116,25...",0.596132,"(1673, T)",1673,T
53,Human,1671,.,GCTCCC,G,12978,PASS,"DP=7755;AF=0.596132;SB=1;DP4=1326,1605,2116,25...",0.596132,"(1674, C)",1674,C
53,Human,1671,.,GCTCCC,G,12978,PASS,"DP=7755;AF=0.596132;SB=1;DP4=1326,1605,2116,25...",0.596132,"(1675, C)",1675,C
53,Human,1671,.,GCTCCC,G,12978,PASS,"DP=7755;AF=0.596132;SB=1;DP4=1326,1605,2116,25...",0.596132,"(1676, C)",1676,C
58,Human,1756,.,GT,G,49314,PASS,"DP=8493;AF=0.947604;SB=0;DP4=217,287,3509,4539...",0.947604,"(1757, T)",1757,T


In [14]:
filtered_df[(filtered_df['REF'].str.len()) < (filtered_df['ALT'].str.len())] # insertion instances that matter

,CHROM,POS,ID,REF,ALT,QUAL,FILTER,INFO,AF
4,Human,80,.,A,AC,49314,PASS,"DP=2829;AF=0.977024;SB=0;DP4=64,0,2747,17;INDE...",0.977024
8,Human,237,.,G,GC,49314,PASS,"DP=4988;AF=0.966921;SB=14;DP4=164,21,3988,835;...",0.966921
9,Human,270,.,C,CG,49314,PASS,"DP=5712;AF=0.945553;SB=6;DP4=242,88,4122,1279;...",0.945553
10,Human,275,.,C,CG,49314,PASS,"DP=5882;AF=0.937946;SB=16;DP4=327,82,4131,1386...",0.937946
11,Human,284,.,A,AC,49314,PASS,"DP=6096;AF=0.960138;SB=15;DP4=185,91,4284,1569...",0.960138
12,Human,363,.,G,GT,49314,PASS,"DP=8340;AF=0.905516;SB=212;DP4=356,457,4641,29...",0.905516
14,Human,386,.,T,TGGCC,49314,PASS,"DP=8821;AF=0.736424;SB=28;DP4=846,550,3631,286...",0.736424
15,Human,419,.,C,CGT,49314,PASS,"DP=9087;AF=0.925278;SB=58;DP4=376,203,4596,381...",0.925278
16,Human,517,.,G,GC,1342,PASS,"DP=9644;AF=0.940377;SB=9;DP4=335,308,5013,4056...",0.940377
17,Human,522,.,C,CG,80,PASS,"DP=9616;AF=0.933548;SB=4;DP4=371,327,4948,4029...",0.933548


In [15]:
#alt way to read vcf files filtered for variant positions
# filter_vcf('/Users/fionachow/Documents/NYU/CDS/Spring 2024/Research Fair/Hochwagen/Individual Data/ERR3240115/ERR3240115_rDNA.vcf') #similar outputs to the pandas filter

In [16]:
mpileup_file_path = '/Users/fionachow/Documents/NYU/CDS/Spring 2024/Research Fair/Hochwagen/Individual Data/ERR3240115/ERR3240115_rDNA.mpileup' #extracting reference alleles for all Positions

mpileup_df = pd.read_csv(mpileup_file_path, sep='\t', header=None, usecols=[0, 1, 2], names=['CHROM', 'POS', 'REF'])

mpileup_df.head(32)

,CHROM,POS,REF
0,Human,1,G
1,Human,2,C
2,Human,3,T
3,Human,4,G
4,Human,5,A
5,Human,6,C
6,Human,7,A
7,Human,8,C
8,Human,9,G
9,Human,10,C


In [17]:
mpileup_df.shape

(13314, 3)

In [18]:
mpileup_df[mpileup_df['REF'].str.len() > 1] #checking each reference position has 1 base

,CHROM,POS,REF


In [19]:
merged_df = mpileup_df.merge(filtered_df[['POS', 'ALT']], on='POS', how='left') #merge with filtered_df i.e. alt(majority) alleles at each position from variant calling

merged_df.head(563)

,CHROM,POS,REF,ALT
0,Human,1,G,NaN
1,Human,2,C,NaN
2,Human,3,T,NaN
3,Human,4,G,NaN
4,Human,5,A,NaN
...,...,...,...,...
558,Human,559,C,NaN
559,Human,560,C,G
560,Human,561,G,GT
561,Human,562,G,NaN


In [20]:
merged_df['REF'] = merged_df.apply(lambda row: row['ALT'] if pd.notnull(row['ALT']) else row['REF'], axis=1) #replace ref values w alt values if not NaN

merged_df.head(104)

,CHROM,POS,REF,ALT
0,Human,1,G,NaN
1,Human,2,C,NaN
2,Human,3,T,NaN
3,Human,4,G,NaN
4,Human,5,A,NaN
...,...,...,...,...
99,Human,100,C,NaN
100,Human,101,C,NaN
101,Human,102,T,NaN
102,Human,103,G,G


In [21]:
# Merge on the position columns, ensuring that deletions_expanded aligns with merged_df
merged_df = pd.merge(merged_df, deletions_expanded, left_on='POS', right_on='Delete_Position', how='left')
        
display(merged_df)

,CHROM_x,POS_x,REF_x,ALT_x,CHROM_y,POS_y,ID,REF_y,ALT_y,QUAL,FILTER,INFO,AF,Deletions,Delete_Position,Base
0,Human,1,G,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Human,2,C,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Human,3,T,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Human,4,G,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Human,5,A,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13309,Human,13310,T,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
13310,Human,13311,C,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
13311,Human,13312,G,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
13312,Human,13313,C,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [22]:
merged_df['REF_x'] = merged_df.apply(replace_ref, axis=1)

In [ ]:
merged_df.drop(['ALT_x', 'CHROM_y', 'POS_y', 'ID', 'REF_y', 'ALT_y', 'QUAL', 'FILTER', 'INFO', 'AF', 'Deletions', 'Delete_Position', 'Base'] , axis=1, inplace=True)

merged_df

In [34]:
merged_df.rename(columns={'CHROM_x':'CHROM', 'POS_x':'POS', 'REF_x':'REF'}, inplace=True)

merged_df[merged_df['REF']==0].shape #CHECKS

(29, 3)

In [25]:
# merged_df.to_csv('reference.csv', index=False)


In [ ]:
#alt way to get reference enumerated
df_fasta = fasta_to_all_chars_dataframe('/Users/fionachow/Documents/rDNA_prototype_prerRNA_only.fa') #manually deleted first line of header

print(df_fasta.head())
print(df_fasta.shape) 

In [ ]:
assert mpileup_df['POS'].equals(df_fasta['POS']) and mpileup_df['REF'].equals(df_fasta['REF']) #checks my workaround matches enumerating reference file
